In [9]:
import pandas as pd
dfriso=pd.read_csv("outputs/per_protein_diamond_data.tsv",sep="\t",index_col="accession")
hgts=set(dfriso[dfriso.tree_HTG].index)

In [10]:
a="""XP_037037647.1
XP_037047887.1
XP_037048891.1
XP_037051484.1
XP_037049572.1
XP_037025072.1
XP_037036945.1
XP_037042570.1
XP_037042572.1
XP_037052143.1
XP_037034543.1
XP_037044115.1
XP_037050080.1
XP_037045845.1
XP_037025508.1
XP_037036450.1
XP_037049686.1
XP_037049098.1
XP_037029925.1
XP_037049483.1
XP_037025913.1
XP_037037225.1
XP_037035924.1
XP_037043806.1
XP_037047695.1
XP_037052058.1
XP_037032840.1
XP_037029512.1"""

In [11]:
cyp450=[x.strip() for x in a.split("\n")]

In [22]:
a="""LOC119075628	XP_037038002.1
LOC119071278	XP_037032039.1
LOC119074162	XP_037036061.1
LOC119074162	XP_037036062.1
LOC119082462	XP_037047875.1
LOC119082463	XP_037047876.1
LOC119085197	XP_037051404.1
LOC119078504	XP_037041958.1
LOC119083332	XP_037048916.1
LOC119083835	XP_037049533.1
LOC119070645	XP_037030969.1
LOC119067246	XP_037026007.1"""
hgtchimeras=[x.split("	")[1].strip() for x in a.split("\n")]
hgtchimeras_map={x.split("	")[1].strip():x.split("	")[0].strip() for x in a.split("\n")}
len(hgtchimeras)

12

In [13]:
a="XP_037024969.1,XP_037027991.1,XP_037028003.1,XP_037049366.1"

In [14]:
tps=[x.strip() for x in a.split(",")]

In [15]:
import pandas as pd
df=pd.DataFrame()
for idx in cyp450:
    df.loc[idx, 'Functional annotation'] = 'Cytochrome P450'
for idx in hgtchimeras:
    df.loc[idx, 'Functional annotation'] = 'Chimeric gene'
for idx in tps:
    df.loc[idx, 'Functional annotation'] = 'Terpene synthase'
df.loc['XP_037029494.1', 'Functional annotation'] = "Tryptophanase/tryptophan-indole lyase (TNase)"
df.loc['XP_037036163.1', 'Functional annotation'] = "Alternative oxidase (AOX)"
df.loc['XP_037034936.1','Functional annotation'] = 'DASH cryptochrome/photolyase'
df.loc['XP_037046651.1','Functional annotation']='Uncharacterized β-propeller gene (propellin)'
df.loc['XP_037030094.1','Functional annotation']='Glycoside hydrolase family 32 (GH32)'
df.loc['XP_037025415.1','Functional annotation']='Glycoside hydrolase family 32 (GH32)'

df.loc['XP_037044875.1','Functional annotation']='Glycosyl hydrase group 43 (GH43)'

df.loc['XP_037050353.1','Functional annotation']='Apoptosis inducing protein 56 (toxin)'
df.loc['XP_037029278.1','Functional annotation']='Lysozyme'
df.loc['XP_037031050.1','Functional annotation']='Bacterial extracellular solute-binding protein'

In [16]:
from Bio import SeqIO
input_file = open(f"datasets/GCF_014529535.1_BU_Bcop_v1_protein.faa")
old_fasta = SeqIO.to_dict(SeqIO.parse(input_file, "fasta"))

input_file = open(f"ncbi_dataset/data/GCF_014529535.2/protein.faa")
new_fasta = SeqIO.to_dict(SeqIO.parse(input_file, "fasta"))
import pandas as pd
new_old=pd.read_csv("outputs/new_vs_old.tsv",sep="\t",header=None)

new_old.columns="qseqid sseqid stitle pident length mismatch gapopen qstart qend sstart send evalue bitscore qcovhsp".split(" ")
new_old=new_old[(new_old['pident']>95)&(new_old['qcovhsp']>85)]
new_old_best = new_old.loc[new_old.groupby("qseqid")["evalue"].idxmin()].reset_index(drop=True)
gff=pd.read_csv("ncbi_dataset/data/GCF_014529535.2/genomic.gff",sep="\t",comment="#",header=None)
def get_gene(x):
    return gff[gff[8].str.contains(x)].iloc[0,8].split(";gene=")[1].split(";")[0]
def old_prot_to_new_prot(x):
    if x in new_fasta.keys():
        return dfriso[dfriso.gene==get_gene(x)].index[0]
    elif x in set(new_old_best.qseqid):
        x= new_old_best[new_old_best.qseqid==x].sseqid.values[0]
        return dfriso[dfriso.gene==get_gene(x)].index[0]
    else:
        return "no_map"


In [17]:
dfriso=pd.read_csv("outputs/per_protein_diamond_data.tsv",sep="\t",index_col="accession")
hgts=set(dfriso[dfriso.tree_HTG].index)

In [18]:
for x, row in df.iterrows():
    an=old_prot_to_new_prot(x)
    print(x,an)
    if an=="no_map":
        df.loc[x,'new_accession']='removed'
    elif an!=x:
        df.loc[x,'new_accession']=an

XP_037037647.1 XP_037037647.1
XP_037047887.1 XP_080113996.1
XP_037048891.1 XP_080119038.1
XP_037051484.1 XP_037051484.1
XP_037049572.1 XP_037049572.1
XP_037025072.1 XP_080120198.1
XP_037036945.1 XP_037036944.1
XP_037042570.1 XP_037042570.1
XP_037042572.1 XP_037042572.1
XP_037052143.1 XP_037052143.1
XP_037034543.1 XP_037034543.1
XP_037044115.1 XP_080112887.1
XP_037050080.1 XP_037050080.1
XP_037045845.1 XP_037045845.1
XP_037025508.1 XP_037025508.1
XP_037036450.1 XP_037036450.1
XP_037049686.1 XP_080118275.1
XP_037049098.1 XP_037049099.1
XP_037029925.1 XP_080112043.1
XP_037049483.1 XP_037049483.1
XP_037025913.1 XP_037025913.1
XP_037037225.1 XP_037037225.1
XP_037035924.1 XP_037035924.1
XP_037043806.1 XP_037043806.1
XP_037047695.1 XP_037047695.1
XP_037052058.1 XP_037052058.1
XP_037032840.1 XP_037032840.1
XP_037029512.1 no_map
XP_037038002.1 XP_080118104.1
XP_037032039.1 no_map
XP_037036061.1 XP_037036061.1
XP_037036062.1 XP_037036061.1
XP_037047875.1 XP_037047875.2
XP_037047876.1 XP_03704787

In [19]:
for index, row in df.iterrows():
    if str(row.new_accession)=='nan':
        n=index
    else:
        n=row.new_accession
        
    if n in set(dfriso[dfriso.tree_HTG].index):
        df.loc[index,'Confirmed']='Yes'
    else:
         df.loc[index,'Confirmed']='No'

In [20]:
df

,Functional annotation,new_accession,Confirmed
XP_037037647.1,Cytochrome P450,NaN,Yes
XP_037047887.1,Cytochrome P450,XP_080113996.1,Yes
XP_037048891.1,Cytochrome P450,XP_080119038.1,Yes
XP_037051484.1,Cytochrome P450,NaN,Yes
XP_037049572.1,Cytochrome P450,NaN,Yes
XP_037025072.1,Cytochrome P450,XP_080120198.1,Yes
XP_037036945.1,Cytochrome P450,XP_037036944.1,Yes
XP_037042570.1,Cytochrome P450,NaN,Yes
XP_037042572.1,Cytochrome P450,NaN,Yes
XP_037052143.1,Cytochrome P450,NaN,Yes


In [23]:
dfc=df[df['Functional annotation']=='Chimeric gene']
for index, row in dfc.iterrows():
    dfc.loc[index,'gene']=hgtchimeras_map[index]

/tmp/ipykernel_974677/3407988024.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfc.loc[index,'gene']=hgtchimeras_map[index]


In [27]:
for i, dfci in dfc.groupby('gene'):
    print(i,dfci)

    print(" ")

LOC119067246                Functional annotation new_accession Confirmed          gene
XP_037026007.1         Chimeric gene           NaN        No  LOC119067246
 
LOC119070645                Functional annotation new_accession Confirmed          gene
XP_037030969.1         Chimeric gene           NaN        No  LOC119070645
 
LOC119071278                Functional annotation new_accession Confirmed          gene
XP_037032039.1         Chimeric gene       removed        No  LOC119071278
 
LOC119074162                Functional annotation   new_accession Confirmed          gene
XP_037036061.1         Chimeric gene             NaN       Yes  LOC119074162
XP_037036062.1         Chimeric gene  XP_037036061.1       Yes  LOC119074162
 
LOC119075628                Functional annotation   new_accession Confirmed          gene
XP_037038002.1         Chimeric gene  XP_080118104.1        No  LOC119075628
 
LOC119078504                Functional annotation   new_accession Confirmed          gene
